# Member 3 — Random Forest

Owner: IT24103815

Purpose: Train and tune a Random Forest classifier.

## Cell 1 — Load the training data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load validated dataset and saved split
data_path = Path("../../data/processed/hotel_bookings_validated_phase3.csv")
split_path = Path("../../results/splits/group_holdout_indices.npz")

df = pd.read_csv(data_path)
splits = np.load(split_path)

# Use the approved 21 features
excluded = [
    "is_canceled", "assigned_room_type", "booking_changes",
    "deposit_type", "agent", "company", "adr",
    "required_car_parking_spaces", "total_of_special_requests",
    "reservation_status", "reservation_status_date"
]

X = df.drop(columns=excluded)
y = df["is_canceled"]

# Reuse the same train/test split
X_train = X.iloc[splits["train_idx"]].copy()
X_test = X.iloc[splits["test_idx"]].copy()

y_train = y.iloc[splits["train_idx"]].copy()
y_test = y.iloc[splits["test_idx"]].copy()

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Selected features:", X_train.shape[1])
print("Training cancellation rate:", f"{y_train.mean():.2%}")

Training shape: (95334, 21)
Testing shape: (23876, 21)
Selected features: 21
Training cancellation rate: 37.01%


## Cell 2 — Import Random Forest and preprocessing tools

In [2]:
import sys

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score

# Import shared preprocessing function
repo_root = data_path.resolve().parents[2]
sys.path.insert(0, str(repo_root))

from src.preprocessing.pipeline import build_preprocessor

print("Random Forest tools imported successfully.")

Random Forest tools imported successfully.


## Cell 3 — Configure Random Forest

In [3]:
# Configure Random Forest classifier
rf_classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=2
)

# Combine preprocessing and classifier
rf_model = Pipeline([
    ("preprocessing", build_preprocessor(X_train)),
    ("classifier", rf_classifier)
])

print("Random Forest pipeline created successfully.")

Random Forest pipeline created successfully.


## Cell 4 — Configure Cross-Validation

In [4]:
# Group identical booking features together
groups_train = pd.util.hash_pandas_object(
    X_train, index=False
)

# Configure 5-fold cross-validation
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Cross-validation configured successfully.")
print("Number of folds:", cv.n_splits)
print("Unique booking feature groups:", groups_train.nunique())

Cross-validation configured successfully.
Number of folds: 5
Unique booking feature groups: 65232


## Cell 5 — Define Evaluation Metrics

In [5]:
# Define evaluation metrics
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

print("Evaluation metrics configured:")
for metric in scoring:
    print("-", metric)

Evaluation metrics configured:
- accuracy
- precision
- recall
- f1
- roc_auc
- pr_auc


## Cell 6 — Train and Evaluate Random Forest

In [6]:
# Train and evaluate Random Forest using 5-fold cross-validation
rf_scores = cross_validate(
    rf_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=True
)

print("Random Forest cross-validation completed successfully.")

Random Forest cross-validation completed successfully.


## Cell 7 — Display Random Forest Evaluation Results

In [7]:
# Display average cross-validation results
print("Random Forest - 5-Fold Cross-Validation\n")

for metric in scoring:
    mean_score = rf_scores[f"test_{metric}"].mean()
    print(f"{metric}: {mean_score:.4f}")

Random Forest - 5-Fold Cross-Validation

accuracy: 0.7877
precision: 0.8899
recall: 0.4851
f1: 0.6277
roc_auc: 0.8832
pr_auc: 0.8402


## Cell 8 — Check Random Forest Overfitting

In [8]:
# Compare training and validation F1-scores
train_f1 = rf_scores["train_f1"].mean()
validation_f1 = rf_scores["test_f1"].mean()

gap = train_f1 - validation_f1

print("Random Forest - Overfitting Check")
print(f"Training F1-score: {train_f1:.4f}")
print(f"Validation F1-score: {validation_f1:.4f}")
print(f"F1-score gap: {gap:.4f}")

Random Forest - Overfitting Check
Training F1-score: 0.6634
Validation F1-score: 0.6277
F1-score gap: 0.0358


## Cell 9 — Save Random Forest Evaluation Results

In [9]:
# Store cross-validation results
results = pd.DataFrame({
    metric: rf_scores[f"test_{metric}"]
    for metric in scoring
})

results.index = [f"Fold {i}" for i in range(1, 6)]
results.loc["Mean"] = results.mean()

display(results.round(4))

# Save results
output = Path("../../results/metrics/random_forest_cv.csv")
output.parent.mkdir(parents=True, exist_ok=True)

results.to_csv(output)

print("Results saved:", output)

,accuracy,precision,recall,f1,roc_auc,pr_auc
Fold 1,0.7910,0.8815,0.4970,0.6356,0.8897,0.8456
Fold 2,0.7874,0.8894,0.5033,0.6428,0.8868,0.8482
Fold 3,0.7862,0.8844,0.4485,0.5951,0.8791,0.8250
Fold 4,0.7927,0.8779,0.4850,0.6248,0.8805,0.8308
Fold 5,0.7811,0.9161,0.4917,0.6399,0.8801,0.8514
Mean,0.7877,0.8899,0.4851,0.6277,0.8832,0.8402


Results saved: ..\..\results\metrics\random_forest_cv.csv


## Random Forest Evaluation Summary

The Random Forest classifier achieved 78.77% accuracy, 88.99% precision, 48.51% recall, and an F1-score of 0.6277 using five-fold group-aware cross-validation.

The model achieved a ROC-AUC of 0.8832 and PR-AUC of 0.8402, demonstrating strong cancellation-risk discrimination.

Although Random Forest produced high precision, its recall was lower than Logistic Regression and Decision Tree. This indicates that the model misses a substantial proportion of actual cancellations at the default classification threshold.

The training F1-score (0.6634) and validation F1-score (0.6277) showed a relatively small performance gap, suggesting limited evidence of overfitting.

Random Forest will be retained for final model comparison. Classification threshold adjustments may be investigated to improve cancellation recall.

## Random Forest — Hyperparameter Tuning

In [10]:
from sklearn.model_selection import RandomizedSearchCV

# Hyperparameters to explore
param_distributions = {
    "classifier__max_depth": [12, 15, 20],
    "classifier__min_samples_leaf": [5, 10, 20],
    "classifier__class_weight": [None, "balanced"]
}

# Group-aware cross-validation
tuning_cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

rf_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_distributions,
    n_iter=6,
    scoring="average_precision",
    cv=tuning_cv,
    random_state=42,
    n_jobs=1
)

print("Random Forest hyperparameter search configured.")
print("Parameter combinations:", rf_search.n_iter)
print("Cross-validation folds:", tuning_cv.n_splits)

Random Forest hyperparameter search configured.
Parameter combinations: 6
Cross-validation folds: 3


In [11]:
# Run Random Forest hyperparameter tuning
rf_search.fit(
    X_train,
    y_train,
    groups=groups_train
)

print("Random Forest tuning completed successfully.")

Random Forest tuning completed successfully.


### Best Hyperparameters and PR-AUC

In [12]:
# Display the best Random Forest tuning results
print("Random Forest - Hyperparameter Tuning Results\n")

print("Best hyperparameters:")
for param, value in rf_search.best_params_.items():
    print(f"{param}: {value}")

print("\nBest cross-validation PR-AUC:",
      f"{rf_search.best_score_:.4f}")

Random Forest - Hyperparameter Tuning Results

Best hyperparameters:
classifier__min_samples_leaf: 5
classifier__max_depth: 15
classifier__class_weight: None

Best cross-validation PR-AUC: 0.8411


In [13]:
# Review all tested Random Forest configurations
tuning_results = pd.DataFrame(rf_search.cv_results_)

columns = [
    "param_classifier__max_depth",
    "param_classifier__min_samples_leaf",
    "param_classifier__class_weight",
    "mean_test_score",
    "std_test_score"
]

display(
    tuning_results[columns]
    .sort_values("mean_test_score", ascending=False)
    .round(4)
)

,param_classifier__max_depth,param_classifier__min_samples_leaf,param_classifier__class_weight,mean_test_score,std_test_score
4,15,5,None,0.8411,0.0083
2,20,20,None,0.8393,0.0097
5,15,10,balanced,0.8376,0.0080
0,12,5,None,0.8346,0.0103
3,15,20,None,0.8344,0.0096
1,12,10,None,0.8323,0.0108


### Tuned Model Evaluation and Comparison

In [14]:
from sklearn.base import clone

# Create the tuned Random Forest model
tuned_rf_model = clone(rf_model)

tuned_rf_model.set_params(
    **rf_search.best_params_
)

print("Tuned Random Forest configured successfully.")
print("Best parameters:", rf_search.best_params_)

Tuned Random Forest configured successfully.
Best parameters: {'classifier__min_samples_leaf': 5, 'classifier__max_depth': 15, 'classifier__class_weight': None}


In [15]:
# Evaluate tuned Random Forest using 5-fold cross-validation
tuned_rf_scores = cross_validate(
    tuned_rf_model,
    X_train,
    y_train,
    groups=groups_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=True
)

print("Tuned Random Forest cross-validation completed successfully.")

Tuned Random Forest cross-validation completed successfully.


In [16]:
# Display tuned Random Forest evaluation results
print("Tuned Random Forest - 5-Fold Cross-Validation\n")

for metric in scoring:
    mean_score = tuned_rf_scores[f"test_{metric}"].mean()
    print(f"{metric}: {mean_score:.4f}")

Tuned Random Forest - 5-Fold Cross-Validation

accuracy: 0.7883
precision: 0.8891
recall: 0.4881
f1: 0.6301
roc_auc: 0.8845
pr_auc: 0.8420


In [17]:
# Compare original and tuned Random Forest results
comparison = pd.DataFrame({
    "Original": {
        metric: rf_scores[f"test_{metric}"].mean()
        for metric in scoring
    },
    "Tuned": {
        metric: tuned_rf_scores[f"test_{metric}"].mean()
        for metric in scoring
    }
}).T

display(comparison.round(4))

,accuracy,precision,recall,f1,roc_auc,pr_auc
Original,0.7877,0.8899,0.4851,0.6277,0.8832,0.8402
Tuned,0.7883,0.8891,0.4881,0.6301,0.8845,0.8420


In [18]:
# Check training and validation performance
train_f1 = tuned_rf_scores["train_f1"].mean()
validation_f1 = tuned_rf_scores["test_f1"].mean()

gap = train_f1 - validation_f1

print("Tuned Random Forest - Overfitting Check")
print(f"Training F1-score: {train_f1:.4f}")
print(f"Validation F1-score: {validation_f1:.4f}")
print(f"F1-score gap: {gap:.4f}")

Tuned Random Forest - Overfitting Check
Training F1-score: 0.6718
Validation F1-score: 0.6301
F1-score gap: 0.0418


### Save Tuning and Validation Results

In [19]:
# Save tuned Random Forest cross-validation results
tuned_results = pd.DataFrame({
    metric: tuned_rf_scores[f"test_{metric}"]
    for metric in scoring
})

tuned_results.index = [
    f"Fold {i}" for i in range(1, len(tuned_results) + 1)
]

tuned_results.loc["Mean"] = tuned_results.mean()

output = Path("../../results/metrics/random_forest_tuned_cv.csv")
output.parent.mkdir(parents=True, exist_ok=True)

tuned_results.round(4).to_csv(output, index_label="Fold")

print("Tuned Random Forest results saved:", output)

Tuned Random Forest results saved: ..\..\results\metrics\random_forest_tuned_cv.csv


In [20]:
# Save Random Forest hyperparameter tuning results
output = Path("../../results/metrics/random_forest_tuning.csv")

tuning_results[[
    "param_classifier__max_depth",
    "param_classifier__min_samples_leaf",
    "param_classifier__class_weight",
    "mean_test_score",
    "std_test_score",
    "rank_test_score"
]].sort_values(
    "rank_test_score"
).to_csv(output, index=False)

print("Random Forest tuning results saved:", output)
print("Best parameters:", rf_search.best_params_)

Random Forest tuning results saved: ..\..\results\metrics\random_forest_tuning.csv
Best parameters: {'classifier__min_samples_leaf': 5, 'classifier__max_depth': 15, 'classifier__class_weight': None}



## Hyperparameter Tuning Summary

Hyperparameter tuning was performed using RandomizedSearchCV with three-fold StratifiedGroupKFold cross-validation, optimizing PR-AUC on the training data.

The selected Random Forest configuration was:
- `max_depth = 15`
- `min_samples_leaf = 5`
- `class_weight = None`

In five-fold cross-validation, tuning slightly improved recall from 0.4851 to 0.4881, F1-score from 0.6277 to 0.6301, and PR-AUC from 0.8402 to 0.8420.

Precision remained high at 0.8891, but recall was relatively low, indicating that many actual cancellations were still missed at the default classification threshold.

The training–validation F1 gap increased from 0.0358 to 0.0418, suggesting a small increase in overfitting.

**Decision:** Retain the tuned Random Forest as a candidate for final model comparison. Prediction-threshold adjustment may be explored to improve cancellation recall. Final model selection will depend on business priorities and evaluation on the untouched test set.
